# 07 · CmbNet — an operator-typed CNN

`CmbNet` stacks operator-typed convolutions (`cmbConv2d`) whose channels are
**differential operators** of the base activation: an `op="grad"` edge layer, an
`op="laplacian"` blob layer, and an `op="band"` scale layer — a learnable
scale-space hierarchy.

We train it on a tiny synthetic orientation task (vertical vs horizontal
edges) and visualise what the operator-typed layers respond to. CPU, seconds.

> Swap the synthetic generator for `torchvision.datasets.MNIST` to run it on
> real MNIST — the model is unchanged.

In [ ]:
import sys
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
from _style import set_style, ACCENT, GOOD, PRIMARY
set_style()

from omnibias.torch.architectures import CmbNet

torch.manual_seed(0)
S = 16  # image size

def make_batch(n):
    """Class 0 = vertical edge, class 1 = horizontal edge (+ noise)."""
    y = torch.randint(0, 2, (n,))
    x = torch.zeros(n, 1, S, S)
    ramp = torch.linspace(-1, 1, S)
    for i in range(n):
        img = ramp.expand(S, S) if y[i] == 0 else ramp.unsqueeze(1).expand(S, S)
        x[i, 0] = img
    x += 0.3 * torch.randn_like(x)
    return x, y

Xtr, Ytr = make_batch(400)
Xte, Yte = make_batch(200)
net = CmbNet(in_channels=1, num_classes=2, width=(8, 16, 16))
print(sum(p.numel() for p in net.parameters()), "parameters")

## Train

In [ ]:
opt = torch.optim.Adam(net.parameters(), lr=3e-3)
loss_hist, acc_hist = [], []
for step in range(120):
    idx = torch.randint(0, Xtr.shape[0], (64,))
    logits = net(Xtr[idx])
    loss = F.cross_entropy(logits, Ytr[idx])
    opt.zero_grad(); loss.backward(); opt.step()
    loss_hist.append(loss.item())
    if step % 10 == 0:
        with torch.no_grad():
            acc = (net(Xte).argmax(1) == Yte).float().mean().item()
        acc_hist.append((step, acc))

with torch.no_grad():
    final_acc = (net(Xte).argmax(1) == Yte).float().mean().item()
print(f"test accuracy = {final_acc:.3f}")

fig, ax = plt.subplots()
ax.plot(loss_hist, color=PRIMARY, label="train loss")
steps, accs = zip(*acc_hist)
ax2 = ax.twinx(); ax2.plot(steps, accs, "o-", color=GOOD, label="test acc"); ax2.set_ylim(0, 1.02)
ax.set_xlabel("step"); ax.set_ylabel("cross-entropy"); ax2.set_ylabel("test accuracy")
ax.set_title("CmbNet training"); plt.show()

## What do the operator-typed layers see?

The first `cmbConv2d` is an `op="grad"` edge layer. We visualise an input from
each class and a few of its edge-response channels.

In [ ]:
with torch.no_grad():
    sample = torch.stack([Xte[(Yte == 0).nonzero()[0, 0]],
                          Xte[(Yte == 1).nonzero()[0, 0]]])
    edges = net.edge(sample)  # (2, C1, S, S)

fig, axes = plt.subplots(2, 4, figsize=(11, 5.4))
for r, label in enumerate(["vertical", "horizontal"]):
    axes[r, 0].imshow(sample[r, 0], cmap="gray"); axes[r, 0].set_ylabel(label)
    axes[r, 0].set_title("input" if r == 0 else "")
    for k in range(3):
        axes[r, k + 1].imshow(edges[r, k], cmap="coolwarm")
        if r == 0:
            axes[r, k + 1].set_title(f"edge ch {k}")
for a in axes.ravel():
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## Takeaway

`CmbNet` reaches high accuracy on the orientation task with a handful of
operator-typed conv layers, and the `op="grad"` layer behaves like a learnable
edge detector. Because `cmbConv2d` is a drop-in for `nn.Conv2d`, you can swap it
into existing CNNs.

Next: **[08 · Keras unified backend](08_keras_unified_backend.ipynb)**.